# RF vs Ridge Walk-Forward — Convergence Experiments

Can we close the RF walk-forward R² gap relative to Ridge?

**Four models compared:**
1. **Ridge WF** — baseline target (R²≈0.456 for earnings)
2. **RF WF (fixed HP)** — current main pipeline (R²≈0.299)
3. **RF WF (adaptive HP)** — retunes `min_samples_leaf` and `max_features`
   every K steps as training data grows
4. **Ridge + RF residual WF** — Ridge captures linear signal, RF fits
   residuals; combines both without sacrificing linear performance

**Requires:** `all_results` from the main pipeline in memory.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

SEED = 42
np.random.seed(SEED)

## Configuration

In [ ]:
# ── Hyperparameter grids ──────────────────────────────────────────────────────
# Full grid — used for initial tuning and adaptive re-tuning
MSL_GRID          = [2, 3, 5, 8, 13, 20, 30]
MAX_FEATURES_GRID = ["sqrt", "log2", 0.05, 0.10]
N_ESTIMATORS      = 400    # slightly reduced from 500 for speed during tuning

# ── Adaptive retuning ─────────────────────────────────────────────────────────
# Retune every RETUNE_EVERY walk-forward steps
# CV_FOLDS folds used for retuning (faster than LOO-CV)
RETUNE_EVERY = 20
CV_FOLDS     = 5

# ── Ridge alpha grid ──────────────────────────────────────────────────────────
RIDGE_ALPHAS = np.logspace(-3, 6, 50)

# ── Which series to run ───────────────────────────────────────────────────────
# Focus on earnings where the gap is meaningful
# Set to list(all_results.keys()) to run both
RUN_SERIES = ["earnings_growth", "dividend_growth"]

print("Config loaded.")

## Shared helpers

In [ ]:
def compute_metrics(y_true, y_pred, y_prev):
    """Return rmse, ar1_rmse, r2, dir_acc for a prediction sequence."""
    sq_err    = (y_true - y_pred) ** 2
    ar1_sq    = (y_true - y_prev) ** 2
    rmse      = np.sqrt(sq_err.mean())
    ar1_rmse  = np.sqrt(ar1_sq.mean())
    r2        = 1 - sq_err.sum() / np.sum((y_true - y_true.mean())**2)
    dy_t      = np.diff(y_true)
    dy_p      = np.diff(y_pred)
    dir_acc   = np.mean(np.sign(dy_t) == np.sign(dy_p)) if len(dy_t) > 0 else np.nan
    return {"rmse": rmse, "ar1_rmse": ar1_rmse, "r2": r2, "dir_acc": dir_acc}


def tune_rf_cv(Z_train, y_train, msl_grid=MSL_GRID,
               mf_grid=MAX_FEATURES_GRID, n_folds=CV_FOLDS):
    """
    Select (min_samples_leaf, max_features) by k-fold CV MSE.
    Uses n_folds rather than LOO for speed during adaptive retuning.
    Falls back to LOO when n_train < 2*n_folds.
    """
    n = len(y_train)
    if n < 2 * n_folds:
        # Too few samples for k-fold — use LOO
        from sklearn.model_selection import LeaveOneOut
        cv = LeaveOneOut()
    else:
        cv = KFold(n_splits=n_folds, shuffle=False)  # no shuffle: time-ordered

    best_mse, best_msl, best_mf = np.inf, msl_grid[0], mf_grid[0]
    for msl in msl_grid:
        for mf in mf_grid:
            preds = np.empty(n)
            for tr, va in cv.split(Z_train):
                m = RandomForestRegressor(
                    n_estimators=100,   # fast during tuning
                    min_samples_leaf=msl, max_features=mf,
                    random_state=SEED, n_jobs=-1,
                )
                m.fit(Z_train[tr], y_train[tr])
                preds[va] = m.predict(Z_train[va])
            mse = mean_squared_error(y_train, preds)
            if mse < best_mse:
                best_mse, best_msl, best_mf = mse, msl, mf
    return best_msl, best_mf


def tune_ridge_gcv(Z_train, y_train, alphas=RIDGE_ALPHAS):
    """Select Ridge alpha by GCV."""
    sc = StandardScaler().fit(Z_train)
    Zs = sc.transform(Z_train)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        rc = RidgeCV(alphas=alphas, gcv_mode="auto",
                     scoring="neg_mean_squared_error")
        rc.fit(Zs, y_train)
    return rc.alpha_, sc


print("Helpers loaded.")

## Model 1 — Ridge walk-forward (target)

In [ ]:
results_ridge = {}

for series_name in RUN_SERIES:
    res       = all_results[series_name]
    Z, y      = res["Z"], res["y"]
    min_train = res["min_train"]
    n         = len(y)
    waves_df  = res["waves_df"]

    print(f"Ridge WF — {series_name}  (steps={n-min_train})")

    # Tune alpha once on initial window
    best_alpha, _ = tune_ridge_gcv(Z[:min_train], y[:min_train])
    print(f"  α={best_alpha:.2e}")

    preds = np.empty(n - min_train)
    for t in range(min_train, n):
        sc_t = StandardScaler().fit(Z[:t])
        m    = Ridge(alpha=best_alpha).fit(sc_t.transform(Z[:t]), y[:t])
        preds[t - min_train] = m.predict(sc_t.transform(Z[t:t+1]))[0]

    y_eval = y[min_train:]
    y_prev = np.concatenate([[y[min_train-1]], y_eval[:-1]])
    m_     = compute_metrics(y_eval, preds, y_prev)
    m_["preds"] = preds
    results_ridge[series_name] = m_

    print(f"  R²={m_['r2']:.3f}  RMSE={m_['rmse']:.4f}  "
          f"AR(1)={m_['ar1_rmse']:.4f}  DirAcc={m_['dir_acc']:.1%}")

## Model 2 — RF walk-forward, fixed hyperparameters

In [ ]:
results_rf_fixed = {}

for series_name in RUN_SERIES:
    res       = all_results[series_name]
    Z, y      = res["Z"], res["y"]
    min_train = res["min_train"]
    n         = len(y)
    bp        = res["best_params"]

    print(f"RF WF fixed HP — {series_name}")
    print(f"  msl={bp['min_samples_leaf']}  mf={bp['max_features']}")

    preds = np.empty(n - min_train)
    for t in range(min_train, n):
        sw = np.log1p(res["waves_df"]["n_after_filter"].values[:t])
        sw = sw / sw.mean()
        m  = RandomForestRegressor(
            n_estimators=N_ESTIMATORS,
            min_samples_leaf=bp["min_samples_leaf"],
            max_features=bp["max_features"],
            random_state=SEED, n_jobs=-1,
        ).fit(Z[:t], y[:t], sample_weight=sw)
        preds[t - min_train] = m.predict(Z[t:t+1])[0]

    y_eval = y[min_train:]
    y_prev = np.concatenate([[y[min_train-1]], y_eval[:-1]])
    m_     = compute_metrics(y_eval, preds, y_prev)
    m_["preds"] = preds
    results_rf_fixed[series_name] = m_

    print(f"  R²={m_['r2']:.3f}  RMSE={m_['rmse']:.4f}  "
          f"AR(1)={m_['ar1_rmse']:.4f}  DirAcc={m_['dir_acc']:.1%}")

## Model 3 — RF walk-forward, adaptive hyperparameters

Retunes `min_samples_leaf` and `max_features` every `RETUNE_EVERY` steps
using k-fold CV on the current training set. Uses 100 trees during tuning
and the full `N_ESTIMATORS` for the actual prediction fits.

In [ ]:
results_rf_adaptive = {}

for series_name in RUN_SERIES:
    res       = all_results[series_name]
    Z, y      = res["Z"], res["y"]
    min_train = res["min_train"]
    n         = len(y)

    print(f"RF WF adaptive HP — {series_name}  "
          f"(retune every {RETUNE_EVERY} steps, {CV_FOLDS}-fold CV)")

    # Initial tuning on first window
    msl, mf = tune_rf_cv(Z[:min_train], y[:min_train])
    print(f"  Initial HP: msl={msl}  mf={mf}")

    preds    = np.empty(n - min_train)
    hp_log   = []   # track how HP evolve

    for t in range(min_train, n):
        step = t - min_train

        # Retune every RETUNE_EVERY steps after the initial window
        if step > 0 and step % RETUNE_EVERY == 0:
            msl, mf = tune_rf_cv(Z[:t], y[:t])
            print(f"  Step {step:>3} (n={t}): retune → msl={msl}  mf={mf}")

        hp_log.append({"step": step, "t": t, "msl": msl, "mf": mf})

        sw = np.log1p(res["waves_df"]["n_after_filter"].values[:t])
        sw = sw / sw.mean()
        m  = RandomForestRegressor(
            n_estimators=N_ESTIMATORS,
            min_samples_leaf=msl, max_features=mf,
            random_state=SEED, n_jobs=-1,
        ).fit(Z[:t], y[:t], sample_weight=sw)
        preds[t - min_train] = m.predict(Z[t:t+1])[0]

    y_eval = y[min_train:]
    y_prev = np.concatenate([[y[min_train-1]], y_eval[:-1]])
    m_     = compute_metrics(y_eval, preds, y_prev)
    m_["preds"]  = preds
    m_["hp_log"] = pd.DataFrame(hp_log)
    results_rf_adaptive[series_name] = m_

    print(f"  R²={m_['r2']:.3f}  RMSE={m_['rmse']:.4f}  "
          f"AR(1)={m_['ar1_rmse']:.4f}  DirAcc={m_['dir_acc']:.1%}")

## Model 4 — Ridge + RF residual walk-forward

Ridge captures the linear signal; RF fits the residuals.
For each walk-forward step:
1. Fit Ridge on training data → get Ridge predictions
2. Compute training residuals
3. Fit RF on training residuals (using adaptive HP)
4. Final prediction = Ridge prediction + RF residual prediction

If the relationship is linear, RF residuals converge to zero and the
combined model equals Ridge. If nonlinearity exists, RF adds incremental R².

In [ ]:
results_ridge_rf = {}

for series_name in RUN_SERIES:
    res       = all_results[series_name]
    Z, y      = res["Z"], res["y"]
    min_train = res["min_train"]
    n         = len(y)

    print(f"Ridge+RF residual WF — {series_name}")

    # Tune Ridge alpha and RF HP once on initial window
    best_alpha, _ = tune_ridge_gcv(Z[:min_train], y[:min_train])
    msl, mf       = tune_rf_cv(Z[:min_train], y[:min_train])
    print(f"  Ridge α={best_alpha:.2e}  RF: msl={msl}  mf={mf}")

    preds_combined = np.empty(n - min_train)
    preds_ridge    = np.empty(n - min_train)
    preds_rf_res   = np.empty(n - min_train)

    for t in range(min_train, n):
        step = t - min_train

        # Retune RF HP every RETUNE_EVERY steps
        if step > 0 and step % RETUNE_EVERY == 0:
            # Compute residuals on current training set for RF tuning
            sc_t    = StandardScaler().fit(Z[:t])
            ridge_t = Ridge(alpha=best_alpha).fit(sc_t.transform(Z[:t]), y[:t])
            resid_t = y[:t] - ridge_t.predict(sc_t.transform(Z[:t]))
            msl, mf = tune_rf_cv(Z[:t], resid_t)

        # Step 1: Ridge
        sc_t      = StandardScaler().fit(Z[:t])
        ridge_fit = Ridge(alpha=best_alpha).fit(sc_t.transform(Z[:t]), y[:t])
        ridge_pred_tr = ridge_fit.predict(sc_t.transform(Z[:t]))
        ridge_pred_va = ridge_fit.predict(sc_t.transform(Z[t:t+1]))[0]

        # Step 2: RF on residuals
        resid_tr = y[:t] - ridge_pred_tr
        sw       = np.log1p(res["waves_df"]["n_after_filter"].values[:t])
        sw       = sw / sw.mean()
        rf_fit   = RandomForestRegressor(
            n_estimators=N_ESTIMATORS,
            min_samples_leaf=msl, max_features=mf,
            random_state=SEED, n_jobs=-1,
        ).fit(Z[:t], resid_tr, sample_weight=sw)
        rf_resid_pred = rf_fit.predict(Z[t:t+1])[0]

        # Step 3: combine
        preds_ridge[step]    = ridge_pred_va
        preds_rf_res[step]   = rf_resid_pred
        preds_combined[step] = ridge_pred_va + rf_resid_pred

    y_eval = y[min_train:]
    y_prev = np.concatenate([[y[min_train-1]], y_eval[:-1]])

    m_combined = compute_metrics(y_eval, preds_combined, y_prev)
    m_ridge_   = compute_metrics(y_eval, preds_ridge, y_prev)
    m_combined["preds"]         = preds_combined
    m_combined["preds_ridge"]   = preds_ridge
    m_combined["preds_rf_res"]  = preds_rf_res
    m_combined["ridge_r2"]      = m_ridge_["r2"]
    results_ridge_rf[series_name] = m_combined

    print(f"  Ridge component R²:   {m_ridge_['r2']:.3f}")
    print(f"  Combined R²:          {m_combined['r2']:.3f}  "
          f"(+{m_combined['r2'] - m_ridge_['r2']:+.3f} from RF residual)")
    print(f"  RMSE={m_combined['rmse']:.4f}  AR(1)={m_combined['ar1_rmse']:.4f}  "
          f"DirAcc={m_combined['dir_acc']:.1%}")

## Summary comparison

In [ ]:
model_results = {
    "Ridge WF":          results_ridge,
    "RF WF (fixed HP)":  results_rf_fixed,
    "RF WF (adaptive)":  results_rf_adaptive,
    "Ridge+RF residual": results_ridge_rf,
}

print("\n── Walk-Forward Summary ──────────────────────────────────────────────────────")
print(f"{'Model':<22} {'Series':<22} {'R²':>8} {'RMSE':>8} {'AR(1)':>8} {'DirAcc':>9}")
print("─" * 82)

for model_name, model_res in model_results.items():
    for series_name in RUN_SERIES:
        if series_name not in model_res: continue
        r = model_res[series_name]
        print(f"{model_name:<22} {series_name:<22} {r['r2']:>8.3f} "
              f"{r['rmse']:>8.4f} {r['ar1_rmse']:>8.4f} {r['dir_acc']:>8.1%}")
    print()

In [ ]:
for series_name in RUN_SERIES:
    res       = all_results[series_name]
    y         = res["y"]
    min_train = res["min_train"]
    waves_df  = res["waves_df"]
    wf_dates  = waves_df["wave_date"].values[min_train:]
    y_eval    = y[min_train:]

    fig, axes = plt.subplots(2, 2, figsize=(15, 9))
    fig.suptitle(f"{series_name.replace('_',' ').title()} — model convergence",
                 fontweight="bold", fontsize=13)

    model_plot = [
        ("Ridge WF",          results_ridge[series_name]["preds"],       "C2"),
        ("RF WF (fixed HP)",  results_rf_fixed[series_name]["preds"],    "C0"),
        ("RF WF (adaptive)",  results_rf_adaptive[series_name]["preds"], "C1"),
        ("Ridge+RF residual", results_ridge_rf[series_name]["preds"],    "C4"),
    ]

    # Panel 1–4: one per model
    for ax, (name, preds, col) in zip(axes.flat, model_plot):
        r2 = compute_metrics(y_eval, preds,
                             np.concatenate([[y[min_train-1]], y_eval[:-1]]))["r2"]
        ax.plot(wf_dates, y_eval, "-", color="black", lw=1.5, label="Realized")
        ax.plot(wf_dates, preds,  "--", color=col, lw=1.5,
                label=f"{name}  (R²={r2:.3f})")
        ax.axvline(wf_dates[0], color="gray", ls=":", alpha=0.5)
        ax.set_title(name); ax.legend(fontsize=8); ax.grid(alpha=0.3)
        ax.set_xlabel("Wave date"); ax.set_ylabel("Forecast")

    plt.tight_layout()
    plt.savefig(f"convergence_{series_name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved convergence_{series_name}.png")

# ── HP evolution plot for adaptive RF ────────────────────────────────────────
for series_name in RUN_SERIES:
    if series_name not in results_rf_adaptive: continue
    hp_log = results_rf_adaptive[series_name]["hp_log"]

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    fig.suptitle(f"Adaptive HP evolution — {series_name.replace('_',' ').title()}",
                 fontweight="bold")

    ax = axes[0]
    ax.step(hp_log["t"], hp_log["msl"], where="post", color="C0", lw=2)
    ax.set_xlabel("Training set size (t)"); ax.set_ylabel("min_samples_leaf")
    ax.set_title("min_samples_leaf over walk-forward steps"); ax.grid(alpha=0.3)

    ax = axes[1]
    mf_numeric = hp_log["max_features"].apply(
        lambda x: float(x) if isinstance(x, float) else
                  {"sqrt": -1, "log2": -2}.get(x, -3)
    )
    ax.step(hp_log["t"], mf_numeric, where="post", color="C1", lw=2)
    ax.set_xlabel("Training set size (t)"); ax.set_ylabel("max_features (encoded)")
    ax.set_title("max_features over walk-forward steps"); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"hp_evolution_{series_name}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ── Rolling RMSE — see which model wins in which regimes ─────────────────────
ROLL = 12

for series_name in RUN_SERIES:
    res       = all_results[series_name]
    y         = res["y"]
    min_train = res["min_train"]
    waves_df  = res["waves_df"]
    wf_dates  = waves_df["wave_date"].values[min_train:]
    y_eval    = y[min_train:]
    y_prev    = np.concatenate([[y[min_train-1]], y_eval[:-1]])

    fig, ax = plt.subplots(figsize=(14, 5))
    fig.suptitle(
        f"Rolling {ROLL}-step RMSE — {series_name.replace('_',' ').title()}",
        fontweight="bold"
    )

    model_plot = [
        ("Ridge WF",          results_ridge[series_name]["preds"],       "C2", "-"),
        ("RF WF (fixed HP)",  results_rf_fixed[series_name]["preds"],    "C0", "--"),
        ("RF WF (adaptive)",  results_rf_adaptive[series_name]["preds"], "C1", "-."),
        ("Ridge+RF residual", results_ridge_rf[series_name]["preds"],    "C4", ":"),
    ]
    for name, preds, col, ls in model_plot:
        sq_err = (y_eval - preds) ** 2
        rolling = pd.Series(sq_err).rolling(ROLL, min_periods=4).mean().pipe(np.sqrt)
        ax.plot(wf_dates, rolling, ls=ls, color=col, lw=1.8, label=name)

    # AR(1) baseline
    ar1_sq = (y_eval - y_prev) ** 2
    ar1_roll = pd.Series(ar1_sq).rolling(ROLL, min_periods=4).mean().pipe(np.sqrt)
    ax.plot(wf_dates, ar1_roll, "-", color="gray", lw=1.5, alpha=0.7,
            label="AR(1) baseline")

    ax.set_xlabel("Wave date"); ax.set_ylabel(f"Rolling {ROLL}-step RMSE")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"rolling_rmse_{series_name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved rolling_rmse_{series_name}.png")